In [13]:
# J = JonesDatabase()
# J.fields(degree=5, root_discriminant_bound=15)
S.<y> = PolynomialRing(QQ)
# d = 14
# (y^6 + 2*y^3 - 1).galois_group().order()
# (y^8 + 2*y^3 - 1).factor()
# (y^3 - y + 6).is_irreducible()
# all((y^n + 4*y^k + 1).is_irreducible() for k in range(1, 30) for n in range(k+1,30)) 
# K = NumberField(y^2 + 3, "b")
# R.<x> = PolynomialRing(GF(3))
# (x^13-1).factor()/
# (x^10 + 3*x + 3).factor()
L = NumberField(y^33 + 2*y^4 + 4, "a")
# K.embeddings(L)
L.discriminant().factor()
# L.galois_group()
# all(len(K.embeddings(NumberField(y^(2*m) + 3*y + 3, "c"))) == 0 for m in range(4, 200)) 
# Mod(-1,53).sqrt()

2^39 * 5 * 39421009 * 93901517 * 914322892059142543843080240112918618877

In [51]:
R.<x> = PolynomialRing(GF(3))
(x^11-1).factor()

(x + 2) * (x^5 + 2*x^3 + x^2 + 2*x + 2) * (x^5 + x^4 + 2*x^3 + x^2 + 2)

In [8]:
S.<a> = PolynomialRing(ZZ)
A = matrix([[1, -1, 1], [2, -1, 0], [-2, -1, 2]])
A.determinant()
-2*A^(-1)
# (A.determinant() - matrix([[-8, 0, 12], [4, 3, 5], [10, 3, 14]]).determinant()).factor()
# matrix([[-8, 0, -12], [4, 3, 5], [10, 3, 14]]).determinant()
# A.determinant().factor()
# TBB = A(3)
# IDBE = matrix([[1,1,1],[1,0,0],[1,1,-1]])
# IDBE * TBB * IDBE^(-1)
# TBB * IDBE^(-1)
# IDBE^(-1)
# -5*(a-3)*(4+a) + 2*(a-5)*(7+2*a) + (a+1)*(10+a)

[-2  1  1]
[-4  4  2]
[-4  3  1]

In [93]:
def check_covering_with_custom_rules(cb, bound, custom_rules=None):
    """
    Check if a 2D covering system covers all coprime pairs, using both a 
    lazy prime callback and a list of explicit custom rules. 
    The callback evaluates residue classes in the range 1...p-1.

    Parameters:
    cb : function(p, a1, a2) -> str, False, or None
        Callback returning a proof string if the prime rule is included.
        Receives a1 and a2 in the range 1...p-1.
    bound : int
        The upper bound for primes to consider.
    custom_rules : list of tuples (m, a1, a2, proof_str)
        A list of explicit rules to apply alongside the prime-based rules.
    """
    if custom_rules is None:
        custom_rules = []
        
    from sage.all import prime_range, gcd, lcm

    primes = [p for p in prime_range(bound + 1) if p - 1 > 0]
    
    # Memoize the callback and handle the 0 -> p-1 mapping here
    memo = {}
    def lazy_cb(p, a1, a2):
        m = p - 1
        # Shift 0 to m (which is p-1)
        mapped_a1 = m if a1 == 0 else a1
        mapped_a2 = m if a2 == 0 else a2
        
        key = (p, mapped_a1, mapped_a2)
        if key not in memo:
            memo[key] = cb(p, mapped_a1, mapped_a2)
        return memo[key]

    def dfs(curr_res_x, curr_res_y, curr_mod):
        chunk_name = f"(x ≡ {curr_res_x}, y ≡ {curr_res_y} mod {curr_mod})"

        # --- PRUNING STEP ---
        g_prune = gcd(curr_res_x, gcd(curr_res_y, curr_mod))
        if g_prune > 1:
            return {
                "type": "pruned",
                "chunk": chunk_name,
                "reason": f"Shared factor {g_prune} (no coprime pairs)"
            }

        # 1. Direct Cover Check
        
        # 1a. Check Custom Rules First
        for rule_m, rule_a1, rule_a2, proof_str in custom_rules:
            if curr_mod % rule_m == 0:
                if curr_res_x % rule_m == (rule_a1 % rule_m) and curr_res_y % rule_m == (rule_a2 % rule_m):
                    return {
                        "type": "covered",
                        "chunk": chunk_name,
                        "rule_source": "custom",
                        "rule": f"(x ≡ {rule_a1}, y ≡ {rule_a2} mod {rule_m})",
                        "proof": proof_str
                    }
                    
        # 1b. Check Prime-based Rules
        for p in primes:
            m = p - 1
            if curr_mod % m == 0:
                a1 = curr_res_x % m
                a2 = curr_res_y % m
                proof_str = lazy_cb(p, a1, a2)
                
                if proof_str not in (None, False):
                    # Format the output to match the 1...p-1 user perspective
                    disp_a1 = m if a1 == 0 else a1
                    disp_a2 = m if a2 == 0 else a2
                    return {
                        "type": "covered",
                        "chunk": chunk_name,
                        "rule_source": f"p={p}",
                        "rule": f"(x ≡ {disp_a1}, y ≡ {disp_a2} mod {m})",
                        "proof": proof_str
                    }

        # 2. Find the best modulus to branch on
        best_m = None
        best_branch_factor = float('inf')

        # 2a. Evaluate Custom Rules for branching
        for rule_m, rule_a1, rule_a2, proof_str in custom_rules:
            if curr_mod % rule_m == 0:
                continue
            
            g = gcd(curr_mod, rule_m)
            
            if (curr_res_x % g == rule_a1 % g) and (curr_res_y % g == rule_a2 % g):
                scale = lcm(curr_mod, rule_m) // curr_mod
                branch_factor = scale * scale
                
                if branch_factor < best_branch_factor:
                    best_branch_factor = branch_factor
                    best_m = rule_m
                    if best_branch_factor == 4:
                        break

        # 2b. Evaluate Prime Rules for branching
        if best_branch_factor > 4: 
            for p in primes:
                m = p - 1
                if curr_mod % m == 0:
                    continue 
                
                g = gcd(curr_mod, m)
                base_a1 = curr_res_x % g
                base_a2 = curr_res_y % g
                has_useful_pair = False
                
                for a1 in range(base_a1, m, g):
                    for a2 in range(base_a2, m, g):
                        if gcd(a1, gcd(a2, m)) > 1:
                            continue

                        if lazy_cb(p, a1, a2) not in (None, False):
                            has_useful_pair = True
                            break
                    if has_useful_pair:
                        break
                
                if has_useful_pair:
                    scale = lcm(curr_mod, m) // curr_mod
                    branch_factor = scale * scale 
                    if branch_factor < best_branch_factor:
                        best_branch_factor = branch_factor
                        best_m = m
                        if best_branch_factor == 4:
                            break

        # 3. Subdivide and Recurse
        if best_m is None:
            return None # Failed to cover

        L = lcm(curr_mod, best_m)
        scale = L // curr_mod
        
        branch_proofs = []
        
        for kx in range(scale):
            next_res_x = curr_res_x + kx * curr_mod
            for ky in range(scale):
                next_res_y = curr_res_y + ky * curr_mod
                
                sub_proof = dfs(next_res_x, next_res_y, L)
                
                if sub_proof is None:
                    return None 
                
                branch_proofs.append(sub_proof)

        return {
            "type": "branched",
            "chunk": chunk_name,
            "split_modulus": L,
            "branches": branch_proofs
        }

    return dfs(0, 0, 1)


def format_proof_tree(proof_node, indent=0):
    """Recursively formats the proof tree into a readable string."""
    if proof_node is None:
        return "Failed to find a complete covering."
        
    pad = "  " * indent
    ptype = proof_node["type"]
    chunk = proof_node["chunk"]
    
    if ptype == "pruned":
        return f"{pad}🚫 {chunk} -> PRUNED: {proof_node['reason']}"
        
    elif ptype == "covered":
        source = proof_node['rule_source']
        src_str = f"Custom Rule" if source == "custom" else f"Prime {source}"
        return f"{pad}✅ {chunk} -> COVERED by {src_str} {proof_node['rule']} (Proof: {proof_node['proof']})"
        
    elif ptype == "branched":
        lines = [f"{pad}🔀 {chunk} -> SPLIT to mod {proof_node['split_modulus']}:"]
        for branch in proof_node["branches"]:
            lines.append(format_proof_tree(branch, indent + 1))
        return "\n".join(lines)

In [94]:
def is_proof_complete(proof_node):
    """
    Recursively scans the proof tree to ensure no branches failed.
    Returns True if fully covered, False if any 'failed' nodes exist.
    """
    if proof_node is None:
        return False
    if proof_node['type'] == 'failed':
        return False
    if proof_node['type'] == 'branched':
        return all(is_proof_complete(child) for child in proof_node['branches'])
    return True


def format_proof_tree(proof_node, indent=0, is_root=True):
    """Recursively formats the proof tree, adding a clear completion banner at the top."""
    if proof_node is None:
        return "No tree generated."
        
    lines = []
    
    # 1. Inject a highly visible banner at the very top of the output
    if is_root:
        if is_proof_complete(proof_node):
            lines.append("==================================================")
            lines.append("🏆 PROOF COMPLETE: All coprime pairs are covered!")
            lines.append("==================================================\n")
        else:
            lines.append("==================================================")
            lines.append("❌ PROOF INCOMPLETE: Found unproven branches.")
            lines.append("==================================================\n")
            
    pad = "  " * indent
    ptype = proof_node["type"]
    chunk = proof_node["chunk"]
    
    # 2. Format the current node
    if ptype == "pruned":
        lines.append(f"{pad}🚫 {chunk} -> PRUNED: {proof_node.get('reason', '')}")
        
    elif ptype == "failed":
        lines.append(f"{pad}❌ {chunk} -> UNPROVEN (FAILED): {proof_node.get('reason', '')}")
        
    elif ptype == "covered":
        source = proof_node.get('rule_source', '')
        src_str = f"Custom Rule" if source == "custom" else f"Prime {source}"
        lines.append(f"{pad}✅ {chunk} -> COVERED by {src_str} {proof_node.get('rule', '')} (Proof: {proof_node.get('proof', '')})")
        
    elif ptype == "branched":
        lines.append(f"{pad}🔀 {chunk} -> SPLIT to mod {proof_node['split_modulus']}:")
        for branch in proof_node["branches"]:
            # Pass is_root=False so the banner only prints once
            lines.append(format_proof_tree(branch, indent + 1, is_root=False))
            
    return "\n".join(lines)


def plot_proof_tree(proof_node):
    """Converts the proof tree into a graph, adding a clear title and print statement for completion."""
    from sage.all import DiGraph
    
    if proof_node is None:
        print("No proof tree to plot.")
        return
        
    # --- Check for completion before doing any heavy graph drawing ---
    complete = is_proof_complete(proof_node)
    
    if complete:
        print("\n" + "🏆 PROOF COMPLETE: All coprime pairs are successfully covered!" + "\n")
        graph_title = "Proof Complete: No Uncovered Branches"
    else:
        print("\n" + "❌ PROOF INCOMPLETE: Look for the red nodes in the graph below." + "\n")
        graph_title = "Proof Incomplete: Contains Failed Branches"
        
    edges = []
    node_labels = {}
    
    colors = {
        '#e0e0e0': [],  # Light Grey for pruned
        '#90ee90': [],  # Green for covered
        '#add8e6': [],  # Blue for branched
        '#ff4d4d': []   # Bright Red for unproven/failed
    }
    
    node_counter = [0]
    
    def traverse(node):
        current_id = node_counter[0]
        node_counter[0] += 1
        
        ptype = node['type']
        chunk = node['chunk']
        
        if ptype == 'pruned':
            reason = node.get('reason', '')
            label = f"[{current_id}] PRUNED\n{chunk}\n{reason}"
            colors['#e0e0e0'].append(label)
            
        elif ptype == 'failed':
            reason = node.get('reason', '')
            label = f"[{current_id}] UNPROVEN\n{chunk}\n{reason}"
            colors['#ff4d4d'].append(label)
            
        elif ptype == 'covered':
            src = node.get('rule_source', '')
            rule = node.get('rule', '')
            proof = node.get('proof', '')
            label = f"[{current_id}] COVERED ({src})\n{chunk}\nRule: {rule}\nProof: {proof}"
            colors['#90ee90'].append(label)
            
        elif ptype == 'branched':
            label = f"[{current_id}] SPLIT mod {node['split_modulus']}\n{chunk}"
            colors['#add8e6'].append(label)
            
        node_labels[current_id] = label
        
        if ptype == 'branched':
            for child in node['branches']:
                child_id = traverse(child)
                edges.append((current_id, child_id))
                
        return current_id
        
    traverse(proof_node)
    
    G = DiGraph(edges)
    G.relabel(node_labels)
    
    plot = G.plot(
        layout='tree',
        tree_root=node_labels[0],
        vertex_colors=colors,
        vertex_size=8000,
        vertex_shape='s',   
        figsize=[20, 15],
        title=graph_title  # Injects the title above the graph image
    )
    
    plot.show()

In [96]:
custom_rules = [
    (2, 1, 0, "Odd degree"),
    (2, 1, 1, "Odd degree")
]

def callback(p, n, k):
    R.<x> = PolynomialRing(QQ)
    K = NumberField(x^2 - 2, "a")
    if any(f.absolute_norm() == p for f in K.primes_above(p)):
        return False
    S.<y> = PolynomialRing(GF(p))
    f = y^n + 2*y^k - 2
    roots = f.roots()
    if len(roots) == 0:
        return None
    return roots

check_covering_with_custom_rules(callback, bound=29, custom_rules=custom_rules)

# is_proof_complete(check_covering_with_custom_rules(callback, bound=100, custom_rules=custom_rules))

{'type': 'branched',
 'chunk': '(x ≡ 0, y ≡ 0 mod 1)',
 'split_modulus': 2,
 'branches': [{'type': 'pruned',
   'chunk': '(x ≡ 0, y ≡ 0 mod 2)',
   'reason': 'Shared factor 2 (no coprime pairs)'},
  {'type': 'covered',
   'chunk': '(x ≡ 0, y ≡ 1 mod 2)',
   'rule_source': 'p=3',
   'rule': '(x ≡ 2, y ≡ 1 mod 2)',
   'proof': [(2, 2)]},
  {'type': 'covered',
   'chunk': '(x ≡ 1, y ≡ 0 mod 2)',
   'rule_source': 'custom',
   'rule': '(x ≡ 1, y ≡ 0 mod 2)',
   'proof': 'Odd degree'},
  {'type': 'covered',
   'chunk': '(x ≡ 1, y ≡ 1 mod 2)',
   'rule_source': 'custom',
   'rule': '(x ≡ 1, y ≡ 1 mod 2)',
   'proof': 'Odd degree'}]}

In [50]:
S.<y> = PolynomialRing(GF(3))
f = y^0 + 2*y^1 + 2
f.roots()
f(0)

0